In [3]:
import numpy as np
import math
import cmath

In [4]:
def parallel_resistance(r1, r2):
    return 1/((1/r1)+(1/r2))

In [6]:
def gain_at_crossover(v_filter, v_ramp, C, Rc, L, Rl, rout, f_crossover):   
    # https://www.ti.com/lit/an/slva301/slva301.pdf?ts=1745660332504
    # Equation 17
    b1 = C*Rc*2*3.1416*f_crossover
    a1 = Rl/rout
    b2 = 2*3.1416*f_crossover*((L/rout)+Rc*C+Rl*C+((Rl*Rc*C)/rout))
    a2 = -(1 + Rc/rout)*L*C*(2*3.1416*f_crossover)**2

    rational = complex(1, b1)/complex(1 + a1 + a2, b2)
    return 1/((v_filter/v_ramp)*cmath.polar(rational)[0])

In [11]:
# Compensation Network Calculations
# https://www.ti.com/lit/an/slva301/slva301.pdf?ts=1745660332504

vout = 5
fsw_post = 300000
iout = 3
actual_inductance = 10*10**-6
effective_capacitance = 247*10**-6
max_pack_voltage = 20
actual_transformer_turns_ratio = 1
effective_esr = 21 *10**-3



vcc = 20
v_ramp = 0.85
vref = 0.696
rbot = 5110 #Ohms
f_crossover = 20000
inductor_dcr = 0.025 #ohms
rout = vout/iout #TODO check which current to use here and/or sweep

a_mod_ccm = vcc/v_ramp
r1 = rbot*((vout/vref)-1)
f_lc = 1/(2*3.1416*math.sqrt(actual_inductance*effective_capacitance))
c3 = 1/(2*3.1416*0.9*f_lc*r1)
r3 = 1/(2*3.1416*f_crossover*c3)
crossover_gain = gain_at_crossover(float(max_pack_voltage)/actual_transformer_turns_ratio, v_ramp, effective_capacitance, effective_esr, actual_inductance, inductor_dcr, rout, f_crossover)
r2 = parallel_resistance(r1, r3)*crossover_gain
c2 = 1/(1.8*3.1416*f_lc*r2)
c1 = 1/(2*3.1416*10*f_crossover*r2)

print(r1)
print(f_lc)
print(c3)
print(r3)
print(crossover_gain)
print(r2)
print(c2)
print(c1)

31599.770114942534
3202.363590947025
1.747519553970692e-09
4553.7278984274535
1.3723178283008226
5462.04635340362
1.0109986734629271e-08
1.456913404077038e-10


In [28]:
fsw = 300*10**3
sandbox_capacitances = np.array([
                [1, 220, 25], 
                [1, 22, 5],
                [50, 0.1, 5]
                ])  # qty, capacitance (uf), esr (mohm)
sandbox_capacitances[:, 1] = sandbox_capacitances[:, 1]*10**-6
sandbox_capacitances[:, 2] = sandbox_capacitances[:, 2]*10**-3
impedances = sandbox_capacitances[:, 2] - 1j/(2*3.1416*fsw*sandbox_capacitances[:, 1])
total_impedance = 1/np.sum((sandbox_capacitances[:, 0]/impedances))
effective_esr = total_impedance.real
effective_capacitance = -1/(total_impedance.imag*2*3.1416*fsw)
print(effective_capacitance*10**6) #uF
print(effective_esr*10**3) #mOhms

51.06923941218819
9.384148660212182


In [2]:
# Isolated Amplifier Calculations
rtot = 3*10**6


def iso_amp(vbus, rtot, r_max_voltage):
    vin_range = 2
    rbot = rtot*((vin_range)/(vbus-vin_range))
    print(rbot)

iso_amp(588, 3*10**6, 200)


10238.907849829351
